In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot  as plt
import seaborn as sns

In [6]:
df=pd.read_csv('D:/DataAnalytics_Projects/Customer churn prediction/Telco-Customer-Churn_dataset.csv')

In [7]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [11]:
#Check the data types and nulls
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [13]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


#### Data Wrangling

In [16]:
#Fix TotalCharges
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
df['TotalCharges'].isna().sum()

11

In [18]:
#Fill those blanks
df['TotalCharges']=df['TotalCharges'].fillna(df['TotalCharges'].median())

In [20]:
#Drop the customerID column
df=df.drop('customerID',axis=1)

In [22]:
# Convert Churn to numeric (0/1) for the model
df['Churn'] = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [30]:
df.to_csv('D:/DataAnalytics_Projects/Customer churn prediction/Telco-Customer-Churn_dataset_cleaned.csv', index=False)

In [32]:
categorical=list(df.select_dtypes(include=['object']).columns)
numerical=list(df.select_dtypes(include=['number']).columns)

In [34]:
categorical

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [36]:
numerical

['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']

In [38]:
#Prepare the features (encode categorical columns)
df_encoded = pd.get_dummies(df, drop_first=True)
df_encoded.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [40]:
#Split into features (X) and target (y)
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

In [42]:
import sklearn

In [46]:
#Split into training and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)
print(X_train.shape, X_test.shape)

(5282, 30) (1761, 30)


In [48]:
#Train the logistic regression model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

C:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [50]:
#Evaluate the model
from sklearn.metrics import classification_report, accuracy_score

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

Accuracy: 81.32%
              precision    recall  f1-score   support

           0       0.85      0.90      0.88      1282
           1       0.69      0.58      0.63       479

    accuracy                           0.81      1761
   macro avg       0.77      0.74      0.75      1761
weighted avg       0.81      0.81      0.81      1761



In [52]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(classification_report(y_test, y_pred))

Accuracy: 81.26%
              precision    recall  f1-score   support

           0       0.85      0.90      0.87      1282
           1       0.68      0.58      0.63       479

    accuracy                           0.81      1761
   macro avg       0.77      0.74      0.75      1761
weighted avg       0.81      0.81      0.81      1761



In [54]:
#Interpret what the model found
import pandas as pd

coefficients = pd.DataFrame({
    'Feature': X_train_scaled.columns,
    'Coefficient': model.coef_[0]
})
coefficients['Abs_Coefficient'] = coefficients['Coefficient'].abs()
coefficients = coefficients.sort_values('Abs_Coefficient', ascending=False)

print(coefficients.head(10))

                           Feature  Coefficient  Abs_Coefficient
25               Contract_Two year    -1.478314         1.478314
1                           tenure    -1.355782         1.355782
10     InternetService_Fiber optic     1.019955         1.019955
3                     TotalCharges     0.674993         0.674993
24               Contract_One year    -0.637558         0.637558
7                 PhoneService_Yes    -0.485912         0.485912
13              OnlineSecurity_Yes    -0.399742         0.399742
2                   MonthlyCharges    -0.370019         0.370019
23             StreamingMovies_Yes     0.358562         0.358562
28  PaymentMethod_Electronic check     0.306787         0.306787


#### Reduces churn (negative coefficients):

Contract_Two year (-1.48) — the single strongest predictor. Two-year contract customers are far less likely to churn than month-to-month customers (the baseline category, since it got dropped by drop_first=True)
tenure (-1.36) — the longer someone's been a customer, the less likely they are to leave
Contract_One year (-0.64) — same effect as two-year, just weaker
OnlineSecurity_Yes (-0.40) — customers with security add-ons churn less, likely because they're more engaged/invested
MonthlyCharges (-0.37) — interesting, this is slightly negative, meaning higher monthly charges alone doesn't predict churn once other factors are controlled for
#### Increases churn (positive coefficients):

InternetService_Fiber optic (+1.02) — fiber customers churn more than DSL customers, even after controlling for other factors — likely pricing or service quality issues
TotalCharges (+0.67) — counterintuitive at first, but this reflects long-tenured high-spenders who eventually churn (it's picking up a different signal than tenure)
PhoneService_Yes (-0.49, note this is negative) — actually reduces churn slightly
StreamingMovies_Yes (+0.36) — customers with streaming add-ons churn slightly more, possibly signaling higher expectations
PaymentMethod_Electronic check (+0.31) — matches what we predicted earlier from the SQL analysis; electronic check users churn more

#### Conclusion
The strongest predictors of churn are contract length and tenure — customers on month-to-month contracts and those with short tenure are significantly more likely to leave. Fiber optic internet customers and those paying via electronic check also show elevated churn risk, suggesting friction with either pricing or payment experience for these segments. Customers with online security add-ons churn less, indicating that increased product engagement correlates with retention

In [60]:
df.to_csv('D:/DataAnalytics_Projects/Telco-Customer-Churn_dataset_cleaned.csv', index=False)
print("Saved cleaned data for SQL and Power BI")

Saved cleaned data for SQL and Power BI
